# Grok-multimodal · FS05-FS06 CLIP + VQA

Dual-encoder contrastive alignment then visual question answering.


In [ ]:
import os, json, math, random, time, re
from pathlib import Path
from collections import Counter
import numpy as np
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path("/kaggle/working"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device", device, "gpus", torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}

def make_shape_image(kind, size=64):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=="red_circle":
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=="blue_square":
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=="green_triangle":
        m=(yy>cy-size*0.25)&(yy<cy+size*0.3)
        m&=np.abs(xx-cx)<(yy-(cy-size*0.25))*0.7; img[m]=(0.15,0.75,0.25)
    elif kind=="yellow_circle":
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.95,0.85,0.1)
    else:
        raise ValueError(kind)
    return img


## FS05 · CLIP-style dual tower


In [ ]:
# CLIP-style dual encoder contrastive learning on synthetic pairs
class ImgEnc(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(64,d)
        )
    def forward(self,x): return F.normalize(self.net(x), dim=-1)

class TxtEnc(nn.Module):
    def __init__(self, d=64, vmax=64):
        super().__init__()
        self.emb=nn.Embedding(vmax, d)
        self.proj=nn.Linear(d,d)
    def forward(self, ids):
        # ids [B,L]
        e=self.emb(ids).mean(1)
        return F.normalize(self.proj(e), dim=-1)

# vocab for clip texts
clip_vocab=["<pad>","a","red","blue","green","yellow","circle","square","triangle","shape"]
clip_stoi={t:i for i,t in enumerate(clip_vocab)}

def tok_text(s, L=6):
    ids=[clip_stoi.get(t,0) for t in s.split()][:L]
    ids+=[0]*(L-len(ids)); return ids

PAIR_BANK=[
    ("red_circle","a red circle"),("blue_square","a blue square"),
    ("green_triangle","a green triangle"),("yellow_circle","a yellow circle"),
]

def make_batch(B=64, size=32):
    imgs=[]; txts=[]; kinds=[]
    for _ in range(B):
        k,t=random.choice(PAIR_BANK)
        img=make_shape_image(k,64)[::2,::2][:size,:size]
        img=np.clip(img+np.random.randn(*img.shape).astype(np.float32)*0.05,0,1)
        imgs.append(img.transpose(2,0,1)); txts.append(tok_text(t)); kinds.append(k)
    return (torch.tensor(np.stack(imgs),dtype=torch.float32),
            torch.tensor(txts,dtype=torch.long), kinds)

img_enc=ImgEnc().to(device); txt_enc=TxtEnc(vmax=len(clip_vocab)).to(device)
opt=torch.optim.Adam(list(img_enc.parameters())+list(txt_enc.parameters()), lr=2e-3)
logit_scale=nn.Parameter(torch.ones([])*np.log(1/0.07))
opt.add_param_group({"params":[logit_scale]})
hist5=[]
for epoch in range(1,21):
    img_enc.train(); txt_enc.train()
    losses=[]; accs=[]
    for _ in range(20):
        xb,tb,_=make_batch(64)
        xb,tb=xb.to(device), tb.to(device)
        zi,zt=img_enc(xb), txt_enc(tb)
        logits=zi@zt.t()*logit_scale.exp()
        labels=torch.arange(len(xb),device=device)
        loss=(F.cross_entropy(logits,labels)+F.cross_entropy(logits.t(),labels))/2
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
        accs.append((logits.argmax(1)==labels).float().mean().item())
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"batch_acc":round(float(np.mean(accs)),4)}
    hist5.append(row)
    if epoch%5==0: print(row)

# retrieval eval
img_enc.eval(); txt_enc.eval()
queries=["a red circle","a blue square","a green triangle","a yellow circle"]
gallery_kinds=["red_circle","blue_square","green_triangle","yellow_circle"]
with torch.no_grad():
    G=torch.stack([torch.tensor(make_shape_image(k,64)[::2,::2][:32,:32].transpose(2,0,1)) for k in gallery_kinds]).float().to(device)
    zg=img_enc(G)
    rows5=[]
    sims_mat=[]
    for q in queries:
        zt=txt_enc(torch.tensor([tok_text(q)],device=device))
        sim=(zt@zg.t()).cpu().numpy()[0]
        sims_mat.append(sim)
        rank=int(sim.argmax())
        rows5.append({"query":q,"top1":gallery_kinds[rank],"scores":[round(float(s),3) for s in sim],
                      "hit":gallery_kinds[rank].replace("_"," ") in q})
print(rows5)
fig,ax=plt.subplots(figsize=(5,4))
im=ax.imshow(np.array(sims_mat),cmap="viridis")
ax.set_xticks(range(4)); ax.set_xticklabels(gallery_kinds,rotation=30,ha="right",fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels(queries,fontsize=8)
ax.set_title("FS05 CLIP-style similarity")
fig.colorbar(im,ax=ax,fraction=0.046); fig.tight_layout()
fig.savefig(FIG/"fs05_clip_sim.png",dpi=120); plt.close()
r1=sum(r["hit"] for r in rows5)/len(rows5)
fs05={"stage":"FS05","method":"dual-encoder InfoNCE (CLIP mini)","history":hist5,"retrieval":rows5,"R@1":r1,
      "vs_prev":"FS03/04 generate text; FS05 learns shared embedding for retrieval",
      "figure":"figures/fs05_clip_sim.png"}
(RES/"fs05.json").write_text(json.dumps(fs05,indent=2)); PROGRESS["FS05"]="ok"; print("FS05 DONE R@1",r1)


## FS06 · VQA fusion


In [ ]:
# VQA: fuse image + question -> answer (closed vocab)
ANS=["red","blue","green","yellow","circle","square","triangle","yes","no"]
ans_stoi={a:i for i,a in enumerate(ANS)}
QBANK=[
    ("red_circle","what color?","red"),
    ("red_circle","what shape?","circle"),
    ("red_circle","is it blue?","no"),
    ("blue_square","what color?","blue"),
    ("blue_square","what shape?","square"),
    ("blue_square","is it a square?","yes"),
    ("green_triangle","what color?","green"),
    ("green_triangle","what shape?","triangle"),
    ("yellow_circle","what color?","yellow"),
    ("yellow_circle","is it a circle?","yes"),
]
q_vocab=["<pad>"]+sorted({w for _,q,_ in QBANK for w in q.replace("?","").split()})
q_stoi={w:i for i,w in enumerate(q_vocab)}

def enc_q(q,L=6):
    toks=q.replace("?","").split()
    ids=[q_stoi.get(t,0) for t in toks][:L]; ids+=[0]*(L-len(ids)); return ids

class VQANet(nn.Module):
    def __init__(self):
        super().__init__()
        self.img=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.qemb=nn.Embedding(len(q_vocab),64)
        self.qproj=nn.Linear(64,64)
        self.head=nn.Sequential(nn.Linear(64+64,128),nn.ReLU(),nn.Linear(128,len(ANS)))
    def forward(self,x,qids):
        fi=self.img(x); fq=self.qproj(self.qemb(qids).mean(1))
        # fusion: concat (classic early VQA); could be Hadamard
        return self.head(torch.cat([fi,fq],-1))

def vqa_batch(B=64,size=32):
    xs,qs,ys=[],[],[]
    for _ in range(B):
        k,q,a=random.choice(QBANK)
        img=make_shape_image(k,64)[::2,::2][:size,:size]
        img=np.clip(img+np.random.randn(*img.shape).astype(np.float32)*0.04,0,1)
        xs.append(img.transpose(2,0,1)); qs.append(enc_q(q)); ys.append(ans_stoi[a])
    return (torch.tensor(np.stack(xs),dtype=torch.float32),
            torch.tensor(qs,dtype=torch.long),
            torch.tensor(ys,dtype=torch.long))

vqa=VQANet().to(device)
opt=torch.optim.Adam(vqa.parameters(), lr=2e-3)
hist6=[]
for epoch in range(1,16):
    vqa.train(); losses=[]; accs=[]
    for _ in range(30):
        xb,qb,yb=vqa_batch(); xb,qb,yb=xb.to(device),qb.to(device),yb.to(device)
        opt.zero_grad(set_to_none=True)
        lg=vqa(xb,qb); loss=F.cross_entropy(lg,yb)
        loss.backward(); opt.step()
        losses.append(loss.item()); accs.append((lg.argmax(1)==yb).float().mean().item())
    row={"epoch":epoch,"loss":round(float(np.mean(losses)),4),"acc":round(float(np.mean(accs)),4)}
    hist6.append(row)
    if epoch%3==0: print(row)

# eval all bank
vqa.eval(); rows6=[]
with torch.no_grad():
    for k,q,a in QBANK:
        img=make_shape_image(k,64)[::2,::2][:32,:32]
        x=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
        qid=torch.tensor([enc_q(q)],device=device)
        pred=ANS[vqa(x,qid).argmax(1).item()]
        rows6.append({"image":k,"question":q,"gt":a,"pred":pred,"ok":pred==a})
acc6=sum(r["ok"] for r in rows6)/len(rows6)
print("VQA acc",acc6,rows6)

# compare: answer from CLIP retrieval of color/shape words only (ablation intuition)
# (simple baseline: if question has color -> majority color name via CLIP text)
fig,axes=plt.subplots(2,5,figsize=(12,5))
for i,(k,q,a) in enumerate(QBANK):
    r=rows6[i]; ax=axes[i//5,i%5]
    ax.imshow(make_shape_image(k)); ax.set_title(f"Q:{q}\nA:{r['pred']} ({'OK' if r['ok'] else 'X'})",fontsize=7); ax.axis("off")
fig.suptitle("FS06 VQA fusion"); fig.tight_layout(); fig.savefig(FIG/"fs06_vqa.png",dpi=120); plt.close()

fs06={"stage":"FS06","method":"CNN+question-emb concat fusion VQA","history":hist6,"acc":acc6,"rows":rows6,
      "vs_prev":"FS05 retrieves similar images; FS06 answers free questions about one image",
      "figure":"figures/fs06_vqa.png"}
(RES/"fs06.json").write_text(json.dumps(fs06,indent=2)); PROGRESS["FS06"]="ok"; print("FS06 DONE")


In [ ]:
summary={"notebook":"Grok-multimodal-fs05-fs06-align-vqa","progress":PROGRESS,"device":str(device)}
(RES/"summary_fs05_fs06.json").write_text(json.dumps(summary,indent=2))
(OUT/"SUCCESS").write_text("ok\n"); print(summary)
